# Mercado de cacau do Sul da Bahia — análise integrada

Este relatório cruza a cotação de cacau comum em Ilhéus, a referência internacional da ICCO, a PTAX, preços e custos da Conab, clima e produção municipal. Valores de baixa frequência não são interpolados como se fossem observações diárias, e correlações não são interpretadas como causalidade.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import polars as pl
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import Markdown, display
pio.renderers.default = 'notebook_connected'
ROOT = Path.cwd()
DATA = ROOT / 'data' / 'processed'
def load(name):
    path = DATA / f'{name}.parquet'
    if not path.exists(): return pd.DataFrame()
    frame = pl.read_parquet(path)
    if frame.columns == ['empty']: return pd.DataFrame()
    return pd.DataFrame(frame.to_dicts())
daily = load('analysis_daily')
weather = load('weather_daily')
producer = load('producer_prices_monthly')
production = load('municipal_production_annual')
costs = load('production_costs')
for frame in [daily, weather, producer, production, costs]:
    for column in frame.columns:
        if column not in {'date','observation_date','reference_month','source','series','market','location_id','municipality_name','municipality_code','product','cost_item','unit','metadata','source_url','timezone','model'}:
            converted = pd.to_numeric(frame[column], errors='coerce')
            if converted.notna().any(): frame[column] = converted
display(Markdown(f'**Gerado com {len(daily):,} datas de mercado, {len(weather):,} observações climáticas e {len(production):,} registros anuais.**'))

## 1. Cobertura e qualidade

In [ ]:
coverage = []
for name, frame, date_col in [('Mercado diário', daily, 'date'), ('Clima', weather, 'observation_date'), ('Preço Conab', producer, 'reference_month'), ('Produção', production, 'year'), ('Custos', costs, 'reference_year')]:
    if frame.empty or date_col not in frame:
        coverage.append((name, 0, None, None, 'sem dados'))
    else:
        series = frame[date_col]
        comparable = frame.map(lambda value: json.dumps(value, sort_keys=True, default=str) if isinstance(value, (dict, list)) else value)
        coverage.append((name, len(frame), str(series.min()), str(series.max()), int(comparable.duplicated().sum())))
coverage_df = pd.DataFrame(coverage, columns=['Conjunto','Linhas','Início','Fim','Duplicatas exatas'])
display(coverage_df)
if not daily.empty:
    display(pd.DataFrame({'ausentes': daily.isna().sum(), 'percentual': daily.isna().mean().mul(100).round(1)}))

## 2. Cotação de Ilhéus, paridade e base local

A paridade bruta é `ICCO (US$/t) × PTAX (R$/US$) × 0,015`. A base é o preço de Ilhéus menos essa paridade; ela resume ágio/deságio, condições domésticas, custos e diferenças de qualidade.

In [ ]:
if daily.empty:
    display(Markdown('> Ainda não há dados diários suficientes. Execute o backfill.'))
else:
    daily['date'] = pd.to_datetime(daily['date'])
    value_cols = [c for c in ['ilheus_brl_arroba','international_parity_brl_arroba'] if c in daily]
    fig = px.line(daily, x='date', y=value_cols, title='Ilhéus versus paridade internacional (R$/arroba)')
    fig.update_layout(legend_title_text='Série', yaxis_title='R$/@')
    fig.show()
    if 'ilheus_basis_brl_arroba' in daily:
        px.line(daily, x='date', y='ilheus_basis_brl_arroba', title='Ágio/deságio de Ilhéus frente à paridade bruta').show()
        display(daily[value_cols + ['ilheus_basis_brl_arroba','ilheus_basis_percent']].describe().round(2))

## 3. Retornos, volatilidade, correlações e defasagens

In [ ]:
if not daily.empty:
    numeric = [c for c in ['ilheus_brl_arroba','icco_usd_tonne','ptax_sell','international_parity_brl_arroba'] if c in daily]
    for col in numeric:
        daily[f'{col}_return'] = pd.to_numeric(daily[col], errors='coerce').pct_change(fill_method=None)
    returns = [f'{c}_return' for c in numeric]
    display(daily[returns].corr().round(3).style.background_gradient(cmap='RdBu', vmin=-1, vmax=1))
    if 'ilheus_brl_arroba_return' in daily and 'international_parity_brl_arroba_return' in daily:
        lagged = []
        for lag in range(-10, 11):
            corr = daily['ilheus_brl_arroba_return'].corr(daily['international_parity_brl_arroba_return'].shift(lag))
            lagged.append({'defasagem_dias': lag, 'correlacao': corr})
        px.bar(pd.DataFrame(lagged), x='defasagem_dias', y='correlacao', title='Correlação defasada: retorno local × paridade').show()
        daily['volatilidade_ilheus_30'] = daily['ilheus_brl_arroba_return'].rolling(30, min_periods=10).std() * (252 ** 0.5)
        px.line(daily, x='date', y='volatilidade_ilheus_30', title='Volatilidade anualizada móvel de 30 pregões').show()

## 4. Sazonalidade e comparação com a Conab

In [ ]:
if not daily.empty and 'ilheus_brl_arroba_return' in daily:
    daily['mes'] = daily['date'].dt.month
    seasonal = daily.groupby('mes')['ilheus_brl_arroba_return'].agg(['mean','median','count']).reset_index()
    px.bar(seasonal, x='mes', y='mean', title='Retorno médio por mês (descritivo)').show()
if not producer.empty:
    display(producer.sort_values('reference_month').tail(24))
else:
    display(Markdown('> A exportação pública mensal da Conab não retornou uma série estruturada nesta execução; a limitação foi registrada no log de ingestão.'))

## 5. Clima local e África Ocidental

In [ ]:
if weather.empty:
    display(Markdown('> Sem dados climáticos.'))
else:
    weather['observation_date'] = pd.to_datetime(weather['observation_date'])
    annual_weather = weather.assign(year=weather['observation_date'].dt.year).groupby(['location_id','year']).agg(
        chuva_mm=('precipitation_sum','sum'),
        temperatura_max_media=('temperature_2m_max','mean'),
        et0_total=('et0_fao_evapotranspiration','sum'),
    ).reset_index()
    px.line(annual_weather, x='year', y='chuva_mm', color='location_id', title='Precipitação anual por região').show()
    px.line(annual_weather, x='year', y='temperatura_max_media', color='location_id', title='Temperatura máxima média anual').show()
    display(Markdown('As séries são reanálises ERA5-Land: representam condições físicas reconstruídas, não a previsão meteorológica disponível ao produtor em cada data.'))

## 6. Produção municipal e custos

In [ ]:
if not production.empty:
    prod = production.copy()
    px.line(prod, x='year', y='production_tonne', color='municipality_name', markers=True, title='Produção municipal de cacau').show()
    px.line(prod, x='year', y='yield_kg_ha', color='municipality_name', markers=True, title='Rendimento municipal').show()
else:
    display(Markdown('> Sem registros municipais estruturados nesta execução.'))
if not costs.empty:
    display(costs.sort_values('reference_year').tail(30))
    display(Markdown('Custos publicados não equivalem ao custo individual da propriedade; qualquer margem derivada é apenas indicativa.'))

## 7. Anomalias, conclusões e próximos passos

In [ ]:
conclusions = []
if not daily.empty and 'ilheus_basis_percent' in daily:
    basis = pd.to_numeric(daily['ilheus_basis_percent'], errors='coerce').dropna()
    if not basis.empty:
        conclusions.append(f'A base local mediana foi {basis.median():.1f}% no período comum; extremos devem ser confrontados com qualidade, importações e condições comerciais.')
if not daily.empty and 'ilheus_brl_arroba_return' in daily:
    r = daily['ilheus_brl_arroba_return'].dropna()
    if len(r) > 10:
        threshold = r.std() * 3
        conclusions.append(f'Foram observados {(r.abs() > threshold).sum()} movimentos diários acima de três desvios-padrão; revisar esses pontos antes de modelar.')
if not conclusions:
    conclusions.append('A cobertura comum ainda é insuficiente para conclusões quantitativas robustas.')
for item in conclusions: display(Markdown(f'- {item}'))
display(Markdown('**Próxima fase:** validar datas de disponibilidade, ampliar o preço físico local, criar baselines e aplicar walk-forward validation sem KFold aleatório.'))